In [0]:
catalog = "vcredit"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

bronze_schema: vcredit.bronze
silver_schema: vcredit.silver
gold_schema: vcredit.gold
landing_path: /Volumes/vcredit/bronze/Landing


### Criação do Catálogo, Schemas e Volume

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.Landing")

print("Catalog, schemas e landing_path criados")    

Catalog, schemas e landing_path criados


## Leitura dos Arquivos da Camada Landing

Nesta etapa, foram definidos os caminhos dos arquivos CSV armazenados na camada Landing e realizada a leitura de cada base utilizando o Spark.


In [0]:
from pyspark.sql.functions import current_timestamp

path_base_atendentes = f"{landing_path}/base_atendentes.csv"
path_base_motivos = f"{landing_path}/base_motivos.csv"
path_canais = f"{landing_path}/canais.csv"
path_chamados = f"{landing_path}/chamados.csv"
path_chamados_hora = f"{landing_path}/Chamados_Hora.CSV"
path_clientes = f"{landing_path}/clientes.csv"
path_custos = f"{landing_path}/custos.csv"
path_pesquisa_satisfacao = f"{landing_path}/pesquisa_satisfacao.csv"

df_base_atendentes_raw = spark.read.csv(path_base_atendentes, header=True, inferSchema=True)
df_base_motivos_raw = spark.read.csv(path_base_motivos, header=True, inferSchema=True)
df_canais_raw = spark.read.csv(path_canais, header=True, inferSchema=True)
df_chamados_raw = spark.read.csv(path_chamados, header=True, inferSchema=True)
df_chamados_hora_raw = spark.read.csv(path_chamados_hora, header=True, inferSchema=True)
df_clientes_raw = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("encoding", "ISO-8859-1")
    .csv(path_clientes)
)
df_custos_raw = spark.read.csv(path_custos, header=True, inferSchema=True)
df_pesquisa_satisfacao_raw = spark.read.csv(path_pesquisa_satisfacao, header=True, inferSchema=True)

## Carga da Camada Bronze

Nesta etapa, os DataFrames lidos da camada Landing foram enriquecidos com a coluna `ingestion_datetime`, responsável por registrar a data e hora da ingestão dos dados.

Em seguida, os dados foram gravados no formato Delta na camada Bronze utilizando o modo `overwrite`, garantindo a atualização completa das tabelas a cada execução do pipeline.

In [0]:
from pyspark.sql.functions import current_timestamp

df_base_atendentes_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.tb_atendentes")

df_base_motivos_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.tb_motivos")

df_canais_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.tb_canais")

df_chamados_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.tb_chamados")

df_chamados_hora_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.tb_chamados_hora")

df_clientes_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.tb_clientes")

df_custos_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.tb_custos")

df_pesquisa_satisfacao_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.tb_pesquisa_satisfacao")

display(spark.table(f"{catalog}.bronze.tb_clientes").limit(10))

id_cliente,nome,email,regiao,idade,data_cadastro,dia_vencimento_fatura,ingestion_datetime
56324416706,Sérgio Martins Miranda,sergio.miranda98@hotmail.com,Sul,51,14/10/2024,10,2026-10-27T14:51:44.719Z
97653229900,Leandro Magalhães Lopes,lopes.leandro@yahoo.com.br,Sudeste,51,2026-07-07,10,2026-10-27T14:51:44.719Z
15132904850,Giovana Soares Vasconcelos,giovana.vasconcelos26@hotmail.com,Sudeste,52,16/02/2019,25,2026-10-27T14:51:44.719Z
62566924227,Adriana Nascimento,adriana96@gmail.com,Nordeste,39,27/08/2025,20,2026-10-27T14:51:44.719Z
35213062014,Vera Duarte,vera.duarte@ig.com.br,Sudeste,34,07/05/2026,15,2026-10-27T14:51:44.719Z
11173485227,Renata Freitas Fernandes,renata96@yahoo.com.br,Sudeste,37,26/01/2025,5,2026-10-27T14:51:44.719Z
50855203771,Lucas Cardoso,lucascardoso@uol.com.br,Nordeste,38,15/07/2026,28,2026-10-27T14:51:44.719Z
72534560781,Camila Guimarães Gomes,camila_gomes@uol.com.br,Nordeste,80,03/09/2025,25,2026-10-27T14:51:44.719Z
40409080977,Thiago Peixoto,thiago98@gmail.com,Sul,92,21/06/2023,25,2026-10-27T14:51:44.719Z
65936173686,Mariana Aparecida Esteves,esteves.mariana@ig.com.br,Norte,41,31/05/2025,15,2026-10-27T14:51:44.719Z
